# Feature Importance Workflow (Steps 3a–3b)

**Purpose:** Run feature importance (Step 3a) and Feature Importance EDA (Step 3b). Run this **after** cohorts exist (see `cohort_workflow.ipynb`).

## Pipeline

1. **Step 3a** – `3a_feature_importance/`: Monte Carlo CV feature importance (CatBoost, XGBoost, XGBoost RF); outputs aggregated feature importances.
2. **Step 3b** – `3b_feature_importance_eda/`: BupaR post-target analysis, code research, refined `cohort_feature_importance.csv` files.

## ICD filtering moved earlier

ICD/administrative code filtering runs in **Step 1b** (`1b_apcd_event_filter`) before cohort creation. That makes data processing more efficient and ensures feature importances (3a/3b) reflect the **same filtered event set**, capturing true predictive features. After rebuilding cohorts with 1b in place, **rerun this workflow** (3a then 3b) so importances are consistent.

## Per-cohort runners

- **3a**: `3a_feature_importance/feature_importance_cohort_runner.ipynb` or `run_mc_feature_importance.py` per cohort/age_band.
- **3b**: Interactive analysis notebooks in `3b_feature_importance_eda/` (e.g. `step3b_interactive_analysis_cohort1.ipynb`).

## Reference

- Secondary guidance: feature importance and model selection (see `3a_feature_importance`, `6_final_model`).
- Shell scripts (archived): `archived/utility_scripts/`; use this notebook instead.

## Configuration

In [ ]:
import sys
import os
from pathlib import Path
import subprocess
import logging

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
if not (PROJECT_ROOT / "3a_feature_importance").exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import (
    sync_s3_to_local,
    check_step_checkpoint_exists,
    save_step_checkpoint,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PYTHON_BIN = Path(sys.executable)
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

COHORTS = {
    "opioid_ed": ["13-24", "25-44", "45-54", "55-64"],
    "non_opioid_ed": ["65-74", "75-84", "85-94"],
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print(f"Python: {PYTHON_BIN}")

## Sync required inputs from S3 to NVMe (idempotent)

Sync **gold/cohorts** from S3 so Step 3a can read cohort parquet from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [ ]:
# Sync gold/cohorts from S3 to local/NVMe (required for 3a feature importance)
s3_cohorts = f"s3://{S3_BUCKET}/gold/cohorts/"
local_cohorts = DATA_ROOT / "gold" / "cohorts"
ok = sync_s3_to_local(s3_cohorts, local_cohorts, profile=AWS_PROFILE)
print(f"  gold/cohorts: {'OK' if ok else 'FAILED or skipped (no AWS CLI)'}")

## Step 3a: MC-CV feature importance — idempotent with checkpoint

Run feature importance for a cohort and age band. Data is read from cohort parquet (synced above to NVMe/local). **Checkpoint:** step is skipped if S3 checkpoint exists for this cohort/age_band.

In [ ]:
# Run Step 3a for one cohort/age_band (idempotent: skip if checkpoint exists)
# Note: 3a script may use its own checkpoint logic; we add notebook-level skip here.
cohort, age_band = "opioid_ed", "13-24"
step_name_3a = "3a_feature_importance"
if check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band} (checkpoint exists). Skipping.")
else:
    result = subprocess.run(
        [str(PYTHON_BIN), "3a_feature_importance/run_mc_feature_importance.py",
         "--cohort", cohort, "--age-band", age_band],
        cwd=PROJECT_ROOT,
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"Exit code: {result.returncode}")

## Step 3b: Feature Importance EDA — idempotent with checkpoint

After 3a, run 3b for BupaR post-target analysis and code research. Refined `cohort_feature_importance.csv` files are produced in `3b_feature_importance_eda/outputs/`. **Checkpoint:** step can be skipped if S3 checkpoint exists. Use the interactive notebooks in `3b_feature_importance_eda/` for full 3b workflow (e.g. `0_icd_cpt_check`, `1_bupaR`).

In [ ]:
# Run Step 3b for one cohort/age_band (idempotent: skip if checkpoint exists)
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "opioid_ed", "13-24"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band} (checkpoint exists). Skipping.")
else:
    # 3b is typically run via EDA workflow script or step3b_interactive_analysis_cohort*.ipynb
    result = subprocess.run(
        [str(PYTHON_BIN), str(STEP3B_DIR / "feature_importance_eda_workflow.py"),
         "--cohort", cohort, "--age-band", age_band],
        cwd=PROJECT_ROOT,
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"Exit code: {result.returncode}")
print(f"Step 3b outputs: {STEP3B_DIR / 'outputs'}")